In [ ]:
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import f1_score, precision_score, accuracy_score, recall_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from scipy.stats import chi2_contingency

In [4]:
# load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [15]:
X_train, y_train = train_df["Combined_Review"], train_df["source"]
X_test,  y_test  = test_df["Combined_Review"],  test_df["source"]
sentiment_test = test_df["Sentiment"]
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [6]:
vectorizer = CountVectorizer()

**RANDOM FOREST CLASSIFIER**

In [7]:
rf_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        min_df=2,
        max_df=0.95
    )),
    ('top_k', SelectKBest(chi2, k=1000)),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer(max_df=0.95, min_df=2)),
                ('top_k',
                 SelectKBest(k=1000,
                             score_func=<function chi2 at 0x000002072CD0E660>)),
                ('rf', RandomForestClassifier(random_state=42))])

In [8]:
param_grid_rf = {
    'vectorizer__min_df': [1, 2, 5],
    'vectorizer__ngram_range': [(1, 1), (1,2)],
    'top_k__k': [500, 1000, 2000, 2500],

    'rf__n_estimators': [200, 500, 750],
    'rf__max_depth': [None, 20],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt']
}

grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('vectorizer',
                                        CountVectorizer(max_df=0.95, min_df=2)),
                                       ('top_k',
                                        SelectKBest(k=1000,
                                                    score_func=<function chi2 at 0x000002072CD0E660>)),
                                       ('rf',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'rf__max_depth': [None, 20],
                         'rf__max_features': ['sqrt'],
                         'rf__min_samples_leaf': [1, 2],
                         'rf__n_estimators': [200, 500, 750],
                         'top_k__k': [500, 1000, 2000, 2500],
                         'vectorizer__min_df': [1, 2, 5],
                         'vectorizer__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy')

In [ ]:
print(grid_rf.best_params_)
# {'rf__max_depth': 20, 
# 'rf__max_features': 'sqrt', 
# 'rf__min_samples_leaf': 1, 
# 'rf__n_estimators': 500, 
# 'top_k__k': 2000, 
# 'vectorizer__min_df': 1, 
# 'vectorizer__ngram_range': (1, 1)}

#{'rf__max_depth': 20, 
# 'rf__max_features': 'sqrt',
#  'rf__min_samples_leaf': 1, 
# 'rf__n_estimators': 500,
#  'top_k__k': 2000,
#  'vectorizer__min_df': 1, 
# 'vectorizer__ngram_range': (1, 1)}

# {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__n_estimators': 500, 'top_k__k': 1000, 'vectorizer__min_df': 1, 'vectorizer__ngram_range': (1, 2)}

best_rf = grid_rf.best_estimator_

{'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__n_estimators': 500, 'top_k__k': 1000, 'vectorizer__min_df': 1, 'vectorizer__ngram_range': (1, 2)}


In [10]:
y_pred_rf = best_rf.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
acc_rf = accuracy_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
pre_rf= precision_score(y_test, y_pred_rf)
print('rf result')
print('f1',f1_rf)
print('-----')
print('accuracy: ', acc_rf)
print('-----')
print('recall:', recall_rf)
print('-----')
print('precision: ', pre_rf)
print('-----')

rf result
f1 0.9240246406570842
-----
accuracy:  0.926
-----
recall: 0.9
-----
precision:  0.9493670886075949
-----


In [11]:
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.95      0.93       250
           1       0.95      0.90      0.92       250

    accuracy                           0.93       500
   macro avg       0.93      0.93      0.93       500
weighted avg       0.93      0.93      0.93       500

[[238  12]
 [ 25 225]]


In [16]:
results = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred_rf,
    'sentiment': sentiment_test
})

for sentiment in ['POS', 'NEG']:
    subset = results[results['sentiment'] == sentiment]

    print(f'\n--- {sentiment} reviews ---')
    print('Number of reviews:', len(subset))
    print('Accuracy:', accuracy_score(
        subset['actual'], subset['predicted']
    ))
    print('Precision:', precision_score(
        subset['actual'], subset['predicted'], zero_division=0
    ))
    print('Recall:', recall_score(
        subset['actual'], subset['predicted'], zero_division=0
    ))
    print('F1:', f1_score(
        subset['actual'], subset['predicted'], zero_division=0
    ))


--- POS reviews ---
Number of reviews: 250
Accuracy: 0.9
Precision: 0.9310344827586207
Recall: 0.864
F1: 0.8962655601659751

--- NEG reviews ---
Number of reviews: 250
Accuracy: 0.952
Precision: 0.9669421487603306
Recall: 0.936
F1: 0.9512195121951219


In [ ]:

results['correct'] = (
    results['actual'] == results['predicted']
)

table = pd.crosstab(
    results['sentiment'],
    results['correct']
)

print(table)

chi2, p_value, dof, expected = chi2_contingency(table)

print('\nChi-square statistic:', chi2)
print('p-value:', p_value)
print('Degrees of freedom:', dof)
print('Expected frequencies:\n', expected)

correct    False  True 
sentiment              
NEG           12    238
POS           25    225

Chi-square statistic: 4.202907010682389
p-value: 0.040354744776541544
Degrees of freedom: 1
Expected frequencies:
 [[ 18.5 231.5]
 [ 18.5 231.5]]


In [14]:
joblib.dump(best_rf, 'models/rf.joblib')

['models/rf.joblib']

**GRADIENT BOOSTING CLASSIFIER**


In [12]:
gb_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        min_df=2,
        max_df=0.95
    )),
    ('top_k', SelectKBest(chi2, k=1000)),
    ('gb', GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ))
])

gb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer(max_df=0.95, min_df=2)),
                ('top_k',
                 SelectKBest(k=1000,
                             score_func=<function chi2 at 0x000002072CD0E660>)),
                ('gb', GradientBoostingClassifier(random_state=42))])

In [13]:
param_grid_gb = {
    'vectorizer__min_df': [1, 2, 5],
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'top_k__k': [500, 1000, 2000],

    'gb__n_estimators': [50, 100, 200],
    'gb__learning_rate': [0.05, 0.1],
    'gb__max_depth': [2, 3],
    'gb__min_samples_leaf': [1, 5],
    'gb__subsample': [0.8, 1.0]
}

grid_gb = GridSearchCV(
    gb_pipeline,
    param_grid_gb,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
print(grid_gb.best_params_)

#{'gb__learning_rate': 0.1, 
# 'gb__max_depth': 3, 
# 'gb__min_samples_leaf': 5, 
# 'gb__n_estimators': 200, 
# 'gb__subsample': 0.8, 
# 'top_k__k': 500, 
# 'vectorizer__min_df': 5, 
# 'vectorizer__ngram_range': (1, 1)}

best_gb = grid_gb.best_estimator_

AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

In [ ]:
y_pred_gb = best_gb.predict(X_test)
f1_gb = f1_score(y_test, y_pred_gb)
acc_gb = accuracy_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
pre_gb= precision_score(y_test, y_pred_gb)
print('gb result')
print('f1',f1_gb)
print('-----')
print('accuracy: ', acc_gb)
print('-----')
print('recall:', recall_gb)
print('-----')
print('precision: ', pre_gb)
print('-----')

gb result
f1 0.9065040650406504
-----
accuracy:  0.908
-----
recall: 0.892
-----
precision:  0.9214876033057852
-----


In [ ]:
joblib.dump(best_gb, 'models/gb.joblib')